# 01 - Exploratory Data Analysis & RDD Operations
This notebook covers environment setup, data cleaning, and core RDD manipulations.

## 0 - Environment Setup

In [1]:
# Install PySpark 
!pip install pyspark

# Install Java 17
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless

# We need openpyxl to read .xlsx files in Python
!pip install openpyxl

Get:1 https://download.docker.com/linux/ubuntu noble InRelease [48.5 kB]
Hit:2 https://packages.cloud.google.com/apt cloud-sdk InRelease                
Hit:3 https://cli.github.com/packages stable InRelease                         
Get:4 https://download.docker.com/linux/ubuntu noble/stable amd64 Packages [68.5 kB]
Hit:5 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Get:6 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://cloud.archive.ubuntu.com/ubuntu noble InRelease [256 kB]         
Get:8 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:9 https://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]     
Get:10 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [2525 kB]
Get:11 http://deb.wakemeops.com/wakemeops stable InRelease [50.5 kB]           
Get:12 https://archive.ubuntu.com/ubuntu noble InRelease [256 kB]              
Get:13 

In [2]:
#!pip install pandas 
#!pip install matplotlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum, avg, when, isnan, round, desc, asc, year, month, to_date, countDistinct, date_sub, abs
from pyspark.sql.types import StringType, IntegerType, DoubleType
from pyspark.sql.functions import max, datediff, lit, to_date
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder, Normalizer, SQLTransformer

# Necessary libraries for Pipeline
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression

In [3]:
import pyspark
print(pyspark.__version__)

3.5.0


In [4]:
spark = SparkSession.builder \
    .appName("BigDataProject") \
    .getOrCreate()

print(spark)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/04 16:39:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1 - Data Cleaning & Preparation

In [5]:
# We use Pandas here only for the initial load, then convert to Spark.

import os
file_path = "../data/online_retail_II.xlsx"
if not os.path.exists(file_path):
    file_path = "online_retail_II.xlsx"
if not os.path.exists(file_path):
    file_path = "data/online_retail_II.xlsx"
if not os.path.exists(file_path):
    file_path = "/teamspace/studios/this_studio/Datasetes/online_retail_II.xlsx"

# Load both sheets
pandas_df_1 = pd.read_excel(file_path, sheet_name="Year 2009-2010", engine="openpyxl")
pandas_df_2 = pd.read_excel(file_path, sheet_name="Year 2010-2011", engine="openpyxl")

# Combine both sheets into one pandas DataFrame
pandas_df = pd.concat([pandas_df_1, pandas_df_2], ignore_index=True)

print("Total rows:", len(pandas_df))
print("Columns:", list(pandas_df.columns))

Total rows: 1067371
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [6]:
pandas_df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394029,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


In [7]:
# Convert Pandas to Spark (no casting yet)
retail_df = spark.createDataFrame(pandas_df)
retail_df = retail_df.withColumnRenamed("Customer ID", "CustomerID") # rename unique identifier column name to "CustomerID"

In [8]:
#Show rows where CustomerID is NULL
print("=== ROWS WHERE CustomerID IS NULL ===")
retail_df.filter(col("CustomerID").isNull()).show(5, truncate=False)

=== ROWS WHERE CustomerID IS NULL ===


26/06/04 16:41:09 WARN TaskSetManager: Stage 0 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.
26/06/04 16:41:10 WARN TaskSetManager: Stage 1 contains a task of very large size (16659 KiB). The maximum recommended task size is 1000 KiB.


+-------+---------+-----------+--------+-----------+-----+----------+-------+
|Invoice|StockCode|Description|Quantity|InvoiceDate|Price|CustomerID|Country|
+-------+---------+-----------+--------+-----------+-----+----------+-------+
+-------+---------+-----------+--------+-----------+-----+----------+-------+



In [9]:
# Checking rows where CustomerID is NaN
print("=== ROWS WHERE CustomerID IS NaN ===")
retail_df.filter(isnan(col("CustomerID"))).show(5, truncate=False)

26/06/04 16:41:11 WARN TaskSetManager: Stage 2 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.


=== ROWS WHERE CustomerID IS NaN ===
+-------+---------+-------------------------+--------+-------------------+-----+----------+--------------+
|Invoice|StockCode|Description              |Quantity|InvoiceDate        |Price|CustomerID|Country       |
+-------+---------+-------------------------+--------+-------------------+-----+----------+--------------+
|489464 |21733    |85123a mixed             |-96     |2009-12-01 10:52:00|0.0  |NaN       |United Kingdom|
|489463 |71477    |short                    |-240    |2009-12-01 10:52:00|0.0  |NaN       |United Kingdom|
|489467 |85123A   |21733 mixed              |-192    |2009-12-01 10:53:00|0.0  |NaN       |United Kingdom|
|489521 |21646    |NaN                      |-50     |2009-12-01 11:44:00|0.0  |NaN       |United Kingdom|
|489525 |85226C   |BLUE PULL BACK RACING CAR|1       |2009-12-01 11:49:00|0.55 |NaN       |United Kingdom|
+-------+---------+-------------------------+--------+-------------------+-----+----------+--------------+


In [10]:
# Count both
null_count = retail_df.filter(col("CustomerID").isNull()).count()
nan_count = retail_df.filter(isnan(col("CustomerID"))).count()

print(f"NULL CustomerIDs: {null_count}")
print(f"NaN  CustomerIDs: {nan_count}")
print(f"Total rows:   {null_count + nan_count}")

26/06/04 16:41:12 WARN TaskSetManager: Stage 3 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.
26/06/04 16:41:14 WARN TaskSetManager: Stage 6 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.


NULL CustomerIDs: 0
NaN  CustomerIDs: 243007
Total rows:   243007


In [11]:
# Remove rows where CustomerID is NaN or null
retail_df = retail_df.filter(
    col("CustomerID").isNotNull() & ~isnan(col("CustomerID"))
)

In [12]:
# Now safe for casting: CustomerID as a string (unique identifier)
retail_df = retail_df.withColumn(
    "CustomerID",
    col("CustomerID").cast("integer").cast("string")
)

print("Rows after cleaning:", retail_df.count())
retail_df.printSchema()

26/06/04 16:41:15 WARN TaskSetManager: Stage 9 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.


Rows after cleaning: 824364
root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)



In [13]:
# Identify nulls for all columns in a single pass
print("=== NULL VALUES PER COLUMN ===")
null_counts_df = retail_df.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in retail_df.columns
])

null_counts_df.show()

=== NULL VALUES PER COLUMN ===


26/06/04 16:41:17 WARN TaskSetManager: Stage 12 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.


+-------+---------+-----------+--------+-----------+-----+----------+-------+
|Invoice|StockCode|Description|Quantity|InvoiceDate|Price|CustomerID|Country|
+-------+---------+-----------+--------+-----------+-----+----------+-------+
|      0|        0|          0|       0|          0|    0|         0|      0|
+-------+---------+-----------+--------+-----------+-----+----------+-------+



In [14]:
print("Clean rows:", retail_df.count())

26/06/04 16:41:18 WARN TaskSetManager: Stage 15 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.


Clean rows: 824364


#### Data cleaning summary:
* Original rows: 1,067,371
* Rows removed: 243,007 (customers with no ID — cannot be used for customer analysis)
* Clean rows: 824,364
* These removed rows represent guest/anonymous transactions with no CustomerID.
* In a real big data pipeline, these could be stored separately for aggregate analysis.

Number of cancellations/returned items (Invoice startswith(C) and Quantity<0):

In [15]:
# How many cancellations/returned items?
returns_df = retail_df.filter(
    col("Invoice").startswith("C") | (col("Quantity") < 0)
)
print("Total cancelled/returned transactions:", returns_df.count())

26/06/04 16:41:19 WARN TaskSetManager: Stage 18 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.


Total cancelled/returned transactions: 18744


Which products are returned the most?

In [16]:
# Which products are returned the most?
top_returns = returns_df.groupBy("StockCode", "Description") \
    .agg(sum("Quantity").alias("Net_Returned_Quantity")) \
    .withColumn("Absolute_Returns", abs(col("Net_Returned_Quantity"))) \
    .orderBy(desc("Absolute_Returns")) \
    .limit(10)

print("=== TOP 10 MOST RETURNED PRODUCTS ===")
top_returns.select("Description", "Absolute_Returns").show(truncate=False)

=== TOP 10 MOST RETURNED PRODUCTS ===


26/06/04 16:41:20 WARN TaskSetManager: Stage 21 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.


+-----------------------------------+----------------+
|Description                        |Absolute_Returns|
+-----------------------------------+----------------+
|PAPER CRAFT , LITTLE BIRDIE        |80995           |
|MEDIUM CERAMIC TOP STORAGE JAR     |74494           |
|ROTATING SILVER ANGELS T-LIGHT HLDR|18730           |
|SET/6 FRUIT SALAD PAPER CUPS       |7140            |
|SET/6 FRUIT SALAD  PAPER PLATES    |7008            |
|Manual                             |5311            |
|POP ART PEN CASE & PENS            |5184            |
|BLACK SILVER FLOWER T-LIGHT HOLDER |5040            |
|MULTICOLOUR SPRING FLOWER MUG      |4996            |
|TEATIME PEN CASE & PENS            |4632            |
+-----------------------------------+----------------+



#### Duplicates analysis:

In [17]:
# Check for duplicate rows
total_rows = retail_df.count()
distinct_rows = retail_df.distinct().count()

print(f"Total rows:    {total_rows}")
print(f"Distinct rows: {distinct_rows}")
print(f"Duplicates:    {total_rows - distinct_rows}")

26/06/04 16:41:22 WARN TaskSetManager: Stage 24 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.
26/06/04 16:41:23 WARN TaskSetManager: Stage 27 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.


Total rows:    824364
Distinct rows: 797885
Duplicates:    26479


In [18]:
# Find duplicate rows by grouping on all columns and counting
duplicates_df = retail_df.groupBy(retail_df.columns).count() \
    .filter(col("count") > 1) \
    .orderBy(col("count").desc())

duplicates_df.show(20)

26/06/04 16:41:27 WARN TaskSetManager: Stage 33 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.


+-------+---------+--------------------+--------+-------------------+-----+----------+--------------+-----+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|CustomerID|       Country|count|
+-------+---------+--------------------+--------+-------------------+-----+----------+--------------+-----+
| 555524|    22698|PINK REGENCY TEAC...|       1|2011-06-05 11:37:00| 2.95|     16923|United Kingdom|   20|
| 555524|    22697|GREEN REGENCY TEA...|       1|2011-06-05 11:37:00| 2.95|     16923|United Kingdom|   12|
| 537224|    70007|HI TEC ALPINE HAN...|       1|2010-12-05 16:24:00| 1.65|     13174|United Kingdom|   10|
| 572861|    22775|PURPLE DRAWERKNOB...|      12|2011-10-26 12:46:00| 1.25|     14102|United Kingdom|    8|
| 537196|    21811|CHRISTMAS HANGING...|       1|2010-12-05 13:55:00| 1.25|     15426|United Kingdom|    6|
| 502660|    17021|NAMASTE SWAGAT IN...|       6|2010-03-25 17:18:00|  0.3|     13187|United Kingdom|    6|
| 536874|    22866|HAND WARM

In [19]:
duplicate_rows = retail_df.groupBy(retail_df.columns).count() \
    .filter(col("count") > 1) \
    .drop("count")  # remove the count column

# Show top 20
duplicate_rows.show(20, truncate=False)

26/06/04 16:41:30 WARN TaskSetManager: Stage 36 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.


+-------+---------+-----------------------------------+--------+-------------------+-----+----------+--------------+
|Invoice|StockCode|Description                        |Quantity|InvoiceDate        |Price|CustomerID|Country       |
+-------+---------+-----------------------------------+--------+-------------------+-----+----------+--------------+
|493706 |82486    |WOOD S/3 CABINET ANT WHITE FINISH  |2       |2010-01-05 14:32:00|7.95 |15768     |United Kingdom|
|494239 |21703    |BAG 125g SWIRLY MARBLES            |1       |2010-01-12 15:48:00|0.42 |14081     |United Kingdom|
|494638 |22367    |CHILDS APRON SPACEBOY DESIGN       |5       |2010-01-17 11:03:00|1.95 |16782     |United Kingdom|
|495311 |22112    |CHOCOLATE HOT WATER BOTTLE         |1       |2010-01-22 14:19:00|4.95 |17243     |United Kingdom|
|495718 |48189    |DOOR MAT FRIENDSHIP                |1       |2010-01-26 14:25:00|6.75 |16791     |United Kingdom|
|496551 |22193    |RED DINER WALL CLOCK               |1       |

In [20]:
# Show duplicate rows sorted by Invoice number
duplicate_rows.orderBy(col("Invoice").asc()).show(5, truncate=False) 

26/06/04 16:41:32 WARN TaskSetManager: Stage 39 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.


+-------+---------+---------------------------------+--------+-------------------+-----+----------+--------------+
|Invoice|StockCode|Description                      |Quantity|InvoiceDate        |Price|CustomerID|Country       |
+-------+---------+---------------------------------+--------+-------------------+-----+----------+--------------+
|489517 |21821    |GLITTER STAR GARLAND WITH BELLS  |1       |2009-12-01 11:34:00|3.75 |16329     |United Kingdom|
|489517 |22319    |HAIRCLIPS FORTIES FABRIC ASSORTED|12      |2009-12-01 11:34:00|0.65 |16329     |United Kingdom|
|489517 |21491    |SET OF THREE VINTAGE GIFT WRAPS  |1       |2009-12-01 11:34:00|1.95 |16329     |United Kingdom|
|489517 |22130    |PARTY CONE CHRISTMAS DECORATION  |6       |2009-12-01 11:34:00|0.85 |16329     |United Kingdom|
|489517 |84951A   |S/4 PISTACHIO LOVEBIRD COASTERS  |1       |2009-12-01 11:34:00|2.55 |16329     |United Kingdom|
+-------+---------+---------------------------------+--------+------------------

In [21]:
# Show a specific invoice that has duplicates so we can see it clearly
duplicate_rows.orderBy(col("Invoice").asc()).limit(1).show(1, truncate=False)

26/06/04 16:41:34 WARN TaskSetManager: Stage 42 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.


+-------+---------+------------------------------+--------+-------------------+-----+----------+--------------+
|Invoice|StockCode|Description                   |Quantity|InvoiceDate        |Price|CustomerID|Country       |
+-------+---------+------------------------------+--------+-------------------+-----+----------+--------------+
|489517 |21913    |VINTAGE SEASIDE JIGSAW PUZZLES|1       |2009-12-01 11:34:00|3.75 |16329     |United Kingdom|
+-------+---------+------------------------------+--------+-------------------+-----+----------+--------------+



In [22]:
# Now find ALL rows with that same invoice number in the original data
first_duplicate_invoice = duplicate_rows.orderBy(col("Invoice").asc()) \
    .first()["Invoice"]

retail_df.filter(col("Invoice") == first_duplicate_invoice) \
    .show(truncate=False)

26/06/04 16:41:36 WARN TaskSetManager: Stage 45 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.


+-------+---------+-----------------------------------+--------+-------------------+-----+----------+--------------+
|Invoice|StockCode|Description                        |Quantity|InvoiceDate        |Price|CustomerID|Country       |
+-------+---------+-----------------------------------+--------+-------------------+-----+----------+--------------+
|489517 |21705    |BAG 500g SWIRLY MARBLES            |1       |2009-12-01 11:34:00|1.65 |16329     |United Kingdom|
|489517 |21913    |VINTAGE SEASIDE JIGSAW PUZZLES     |1       |2009-12-01 11:34:00|3.75 |16329     |United Kingdom|
|489517 |21912    |VINTAGE SNAKES & LADDERS           |1       |2009-12-01 11:34:00|3.75 |16329     |United Kingdom|
|489517 |16207A   |PINK STRAWBERRY HANDBAG            |2       |2009-12-01 11:34:00|2.95 |16329     |United Kingdom|
|489517 |21821    |GLITTER STAR GARLAND WITH BELLS    |1       |2009-12-01 11:34:00|3.75 |16329     |United Kingdom|
|489517 |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |1       |

26/06/04 16:41:38 WARN TaskSetManager: Stage 48 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.


In [23]:
# Find only the rows that are exact duplicates and show them side by side
retail_df.groupBy(retail_df.columns) \
    .count() \
    .filter(col("count") > 1) \
    .orderBy(col("count").desc()) \
    .show(5, truncate=False)

26/06/04 16:41:39 WARN TaskSetManager: Stage 49 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.


+-------+---------+-----------------------------------+--------+-------------------+-----+----------+--------------+-----+
|Invoice|StockCode|Description                        |Quantity|InvoiceDate        |Price|CustomerID|Country       |count|
+-------+---------+-----------------------------------+--------+-------------------+-----+----------+--------------+-----+
|555524 |22698    |PINK REGENCY TEACUP AND SAUCER     |1       |2011-06-05 11:37:00|2.95 |16923     |United Kingdom|20   |
|555524 |22697    |GREEN REGENCY TEACUP AND SAUCER    |1       |2011-06-05 11:37:00|2.95 |16923     |United Kingdom|12   |
|537224 |70007    |HI TEC ALPINE HAND WARMER          |1       |2010-12-05 16:24:00|1.65 |13174     |United Kingdom|10   |
|572861 |22775    |PURPLE DRAWERKNOB ACRYLIC EDWARDIAN|12      |2011-10-26 12:46:00|1.25 |14102     |United Kingdom|8    |
|502660 |17021    |NAMASTE SWAGAT INCENSE             |6       |2010-03-25 17:18:00|0.3  |13187     |United Kingdom|6    |
+-------+-------

In [24]:
# Show all 20 copies of this specific duplicate row
retail_df.filter(
    (col("Invoice") == "555524") & 
    (col("StockCode") == "22698")
).show(25, truncate=False)

26/06/04 16:41:41 WARN TaskSetManager: Stage 52 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.
26/06/04 16:41:41 WARN TaskSetManager: Stage 53 contains a task of very large size (16659 KiB). The maximum recommended task size is 1000 KiB.


+-------+---------+------------------------------+--------+-------------------+-----+----------+--------------+
|Invoice|StockCode|Description                   |Quantity|InvoiceDate        |Price|CustomerID|Country       |
+-------+---------+------------------------------+--------+-------------------+-----+----------+--------------+
|555524 |22698    |PINK REGENCY TEACUP AND SAUCER|1       |2011-06-05 11:37:00|2.95 |16923     |United Kingdom|
|555524 |22698    |PINK REGENCY TEACUP AND SAUCER|1       |2011-06-05 11:37:00|2.95 |16923     |United Kingdom|
|555524 |22698    |PINK REGENCY TEACUP AND SAUCER|1       |2011-06-05 11:37:00|2.95 |16923     |United Kingdom|
|555524 |22698    |PINK REGENCY TEACUP AND SAUCER|1       |2011-06-05 11:37:00|2.95 |16923     |United Kingdom|
|555524 |22698    |PINK REGENCY TEACUP AND SAUCER|1       |2011-06-05 11:37:00|2.95 |16923     |United Kingdom|
|555524 |22698    |PINK REGENCY TEACUP AND SAUCER|1       |2011-06-05 11:37:00|2.95 |16923     |United K

In [25]:
# Remove all duplicates
retail_df = retail_df.distinct().cache()
print("Final clean rows:", retail_df.count())

26/06/04 16:41:42 WARN TaskSetManager: Stage 54 contains a task of very large size (16645 KiB). The maximum recommended task size is 1000 KiB.


Final clean rows: 797885


In [26]:
# Remove all duplicate rows — keep only one copy of each
retail_df = retail_df.distinct()
print("Final clean rows:", retail_df.count())

Final clean rows: 797885


In [27]:
# Confirm our clean dataset looks right
retail_df.show(5, truncate=False)

+-------+---------+----------------------------------+--------+-------------------+-----+----------+--------------+
|Invoice|StockCode|Description                       |Quantity|InvoiceDate        |Price|CustomerID|Country       |
+-------+---------+----------------------------------+--------+-------------------+-----+----------+--------------+
|489465 |20747    |PICCADILLY TEA SET                |4       |2009-12-01 10:52:00|14.95|13767     |United Kingdom|
|489488 |20972    |PINK CREAM FELT CRAFT TRINKET BOX |3       |2009-12-01 10:59:00|1.25 |17238     |United Kingdom|
|489522 |85044    |RED REINDEER STRING OF 20 LIGHTS  |2       |2009-12-01 11:45:00|4.95 |15998     |United Kingdom|
|489522 |21948    |SET OF 6 CAKE CHOPSTICKS          |6       |2009-12-01 11:45:00|1.25 |15998     |United Kingdom|
|489522 |79341    |WILLOW BRANCH LIGHTS.             |6       |2009-12-01 11:45:00|6.75 |15998     |United Kingdom|
+-------+---------+----------------------------------+--------+---------

### Q1c — Cancellation / return rate per product

In [28]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [29]:
# Split sales vs returns 
sales_df   = retail_df.filter(F.col("Quantity") > 0)
returns_df = retail_df.filter(F.col("Quantity") < 0)

# Get the most common description per StockCode
desc_window = Window.partitionBy("StockCode").orderBy(F.desc("freq"))

description_df = (
    retail_df
    .groupBy("StockCode", "Description")
    .agg(F.count("*").alias("freq"))
    .withColumn("rn", F.row_number().over(desc_window))
    .filter(F.col("rn") == 1)
    .select("StockCode", "Description")
)

# Total units sold per StockCode only 
total_sold = (
    sales_df
    .groupBy("StockCode")                          # removed Description
    .agg(F.sum("Quantity").alias("total_sold"))
)

# Total units returned per StockCode 
total_returned = (
    returns_df
    .groupBy("StockCode")
    .agg(
        F.abs(F.sum("Quantity")).alias("total_returned"),
        F.countDistinct("Invoice").alias("return_invoices")
    )
)

# Join everything and compute return rate
result = (
    total_sold
    .join(total_returned, on="StockCode", how="left")
    .join(description_df, on="StockCode", how="left")
    .fillna(0, subset=["total_returned", "return_invoices"])
    .filter(F.col("total_sold") > 0)              # exclude zero-sale products
    .withColumn(
        "return_rate_%",
        F.round((F.col("total_returned") / F.col("total_sold")) * 100, 2)
    )
    .select("StockCode", "Description", "total_sold", 
            "total_returned", "return_invoices", "return_rate_%")
    .orderBy(F.col("return_rate_%").desc())
)

result.show(20, truncate=False)

+---------+-----------------------------------+----------+--------------+---------------+-------------+
|StockCode|Description                        |total_sold|total_returned|return_invoices|return_rate_%|
+---------+-----------------------------------+----------+--------------+---------------+-------------+
|D        |Discount                           |196       |3061          |143            |1561.73      |
|35976B   |WHITE SCANDINAVIAN HEART CHRISTMAS |1         |12            |1              |1200.0       |
|79301    |FEATHER HEART LIGHTS               |1         |9             |4              |900.0        |
|85043    |RED HEART CANDY POP LIGHTS         |1         |3             |3              |300.0        |
|37451    |CERAMIC CAKE TEAPOT WITH CHERRY    |1         |2             |2              |200.0        |
|85069    |CREAM SWEETHEART DOUBLE SHELF      |3         |6             |2              |200.0        |
|20879    |TREE OF NOAH FESTIVE SCENTED CANDLE|50        |96    

### Issue: Some products showed return rates > 100%
Root causes:
1. Dataset covers 2009-2011 but returns reference original StockCode regardless of purchase date — returns from pre-2009 purchases appear against 2009-2011 sales only
2. Low-volume products (e.g. sold=1, returned=12) produce statistically meaningless rates
3. Non-product StockCodes (e.g. "D"=Discount) inflate returns

In [30]:
# (minimum volume filter):
# Exclude StockCodes with < 10 units sold (low volume)
# Exclude non-product codes (letters-only StockCodes)
# Cap return rate at 100% (cross-period returns)

result = (
    total_sold
    .join(total_returned, on="StockCode", how="left")
    .join(description_df, on="StockCode", how="left")
    .fillna(0, subset=["total_returned", "return_invoices"])
    .filter(F.col("total_sold") >= 10)        # minimum 10 units sold
    .filter(~F.col("StockCode").rlike("^[A-Za-z]+$"))  # remove non-products
    .withColumn(
        "return_rate_%",
        F.round((F.col("total_returned") / F.col("total_sold")) * 100, 2)
    )
    .filter(F.col("return_rate_%") <= 100)
    .select("StockCode", "Description", "total_sold",
            "total_returned", "return_invoices", "return_rate_%")
    .orderBy(F.col("return_rate_%").desc())
)

result.show(20, truncate=False)

+---------+---------------------------------+----------+--------------+---------------+-------------+
|StockCode|Description                      |total_sold|total_returned|return_invoices|return_rate_%|
+---------+---------------------------------+----------+--------------+---------------+-------------+
|23843    |PAPER CRAFT , LITTLE BIRDIE      |80995     |80995         |1              |100.0        |
|44217M   |MULTICOLOUR FEATHERS CURTAIN     |72        |72            |2              |100.0        |
|20715    |LITTLE FLOWER SHOPPER BAG        |400       |400           |1              |100.0        |
|21496    |WOODLAND CREATURES  WRAP         |25        |25            |1              |100.0        |
|21950    |SET OF 6 DOTS CHOPSTICKS         |185       |181           |2              |97.84        |
|23166    |MEDIUM CERAMIC TOP STORAGE JAR   |77916     |74494         |10             |95.61        |
|72045D   |ROSES ON BLUE TEACUP CANDLE      |536       |504           |1          

In [31]:
# Remove cancelled orders (Invoice starting with 'C') — these are refunds
# Remove negative or zero quantities — not valid purchases
# Remove zero or negative prices — not valid transactions

retail_df = retail_df.filter(
    (~col("Invoice").startswith("C")) &  # remove cancellations
    (col("Quantity") > 0) &              # remove negative/zero quantity
    (col("Price") > 0)                   # remove zero/negative price
)

print("Final clean rows after removing cancellations:", retail_df.count())

Final clean rows after removing cancellations: 779425


### Data Cleaning Summary

| Step | Rows Removed | Reason |
|---|---|---|
| Remove null CustomerIDs | 243,007 | Anonymous transactions — no customer to analyse |
| Remove duplicates | 26,479 | System recording errors in source data |
| Remove cancellations & bad prices | 18,460 | Invoices starting with 'C' are refunds, not sales |
| **Final clean dataset** | **779,425** | Valid purchase transactions ready for analysis |

Note: All cleaning steps are big-data safe and would scale to billions of rows across distributed partitions.

In [32]:
# Convert our clean Spark DataFrame to an RDD
# Each row becomes a Row object in the RDD
retail_rdd = retail_df.rdd

print("Type:", type(retail_rdd))
print("Number of partitions:", retail_rdd.getNumPartitions())
print("First record:")
print(retail_rdd.first())

Type: <class 'pyspark.rdd.RDD'>
Number of partitions: 4
First record:
Row(Invoice='489465', StockCode='20747', Description='PICCADILLY TEA SET', Quantity=4, InvoiceDate=datetime.datetime(2009, 12, 1, 10, 52), Price=14.95, CustomerID='13767', Country='United Kingdom')


In [33]:
# Extract the SparkContext from the SparkSession
sc = spark.sparkContext

range_data = range(0, 10)
simpleRDD = sc.parallelize(range_data, 2)

## 2 - Customer Segmentation & Lifetime Value (RFM)

In [34]:
# RDD TRANSFORMATIONS

# Transform each row into a (CustomerID, revenue) pair
# Revenue = Quantity × Price
customer_revenue_rdd = retail_rdd.map(
    lambda row: (row.CustomerID, row.Quantity * row.Price)
)

# reduceByKey: sum revenue per customer
customer_total_rdd = customer_revenue_rdd.reduceByKey(
    lambda a, b: a + b
)

# sortBy: find top 10 customers by revenue
top_customers_rdd = customer_total_rdd.sortBy(
    lambda x: x[1], ascending=False
)

# Show top 10
print("=== TOP 10 CUSTOMERS BY REVENUE ===")
for customer in top_customers_rdd.take(10):
    print(f"Customer {customer[0]}: €{customer[1]:,.3f}")

=== TOP 10 CUSTOMERS BY REVENUE ===
Customer 18102: €580,987.040
Customer 14646: €528,602.520
Customer 14156: €313,437.620
Customer 14911: €291,420.810
Customer 17450: €244,784.250
Customer 13694: €195,640.690
Customer 17511: €172,132.870
Customer 16446: €168,472.500
Customer 16684: €147,142.770
Customer 12415: €144,458.370


Note: Using reduceByKey instead of groupByKey — reduceByKey performs partial aggregation within each partition before shuffling data across the network, making it significantly more efficient at scale.

In [35]:
# RDD filter — find high value customers (over £10,000 revenue)

high_value_rdd = customer_total_rdd.filter(
    lambda x: x[1] > 10000
)

print("High value customers (>€10,000):", high_value_rdd.count())

High value customers (>€10,000): 261


"Out of all our customers, just 261 account for over £10,000 each in revenue. Identifying and retaining these customers should be the top marketing priority."


In [36]:
# RDD — What % of total revenue do top customers generate?

# Total revenue from ALL customers
total_revenue = customer_total_rdd.map(
    lambda x: x[1]
).sum()

# Total revenue from HIGH VALUE customers only
high_value_revenue = high_value_rdd.map(
    lambda x: x[1]
).sum()

# Calculate percentage
percentage = (high_value_revenue / total_revenue) * 100

print(f"Total revenue:            €{total_revenue:,.2f}")
print(f"High value revenue:       €{high_value_revenue:,.2f}")
print(f"High value customers %:   {percentage:.1f}% of total revenue")
print(f"Number of High-Value customers:   261")

Total revenue:            €17,374,804.27
High value revenue:       €8,714,190.75
High value customers %:   50.2% of total revenue
Number of High-Value customers:   261
